In [3]:
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import numpy as np
import json
import joblib

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
model_uri = "models:/fraud-detection@champion"
model = mlflow.sklearn.load_model(model_uri)

In [3]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import add_all_features

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [7]:
X_train = add_all_features(val)
y_train = val['isFraud']
from model.preprocessor_pipe_evalueate import evaluate_model
X_test = add_all_features(test)
y_test = test['isFraud']
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("final_evaluation")
with mlflow.start_run(run_name="lightgbm_val_training"):

    model.fit(X_train, y_train)
    
    metrics = evaluate_model(model, X_test, y_test)

    mlflow.log_param("model", "lightgbm")

    mlflow.log_param("number_of_features", X_test.shape[1])

    mlflow.log_metrics(metrics)

[LightGBM] [Info] Number of positive: 3042, number of negative: 85539
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6145
[LightGBM] [Info] Number of data points in the train set: 88581, number of used features: 1687
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034341 -> initscore=-3.336457
[LightGBM] [Info] Start training from score -3.336457
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

In [10]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("final_evaluation")
with mlflow.start_run(run_id="0bcc843581c848c89d4afacd52efe027"):
    mlflow.sklearn.log_model(model, artifact_path="model", serialization_format="pickle")

2026/08/29 13:54:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/29 13:54:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lightgbm_val_training at: http://127.0.0.1:5000/#/experiments/7/runs/0bcc843581c848c89d4afacd52efe027
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [4]:
load_dotenv()
model_path = os.getenv("MODEL_PATH")
joblib.dump(model, model_path)

['D:\\IT\\projects\\fraud_detection\\models\\fraud_detection_model.joblib']